In [ ]:
from redbox.graph.root import get_root_graph
from redbox.models.settings import Settings, ElasticLocalSettings
from pathlib import Path
from dotenv import load_dotenv, dotenv_values
from langchain_openai import AzureChatOpenAI
import tiktoken
from core_api.dependencies import get_parameterised_retriever, get_all_chunks_retriever, get_embedding_model
import logging
import statistics

_ = load_dotenv(Path.cwd().parent / ".env")

ENV = Settings(
    minio_host="localhost", 
    object_store="minio", 
    unstructured_host="localhost",
    worker_ingest_min_chunk_size=1_000,
    worker_ingest_max_chunk_size=10_000,
    worker_ingest_largest_chunk_size=300_000,
    elastic=ElasticLocalSettings(host="localhost"),
)

LLM = AzureChatOpenAI(
    api_key=ENV.azure_openai_api_key_4o,
    azure_endpoint=ENV.azure_openai_endpoint_4o,
    model="gpt-4o",
)

TEST_INDEX = "rag-test-index"

In [ ]:
ENV.worker_ingest_min_chunk_size, ENV.worker_ingest_max_chunk_size, ENV.worker_ingest_largest_chunk_size, ENV.worker_ingest_largest_chunk_overlap

# RAG: make it good

By hook or by crook we need to make RAG half decent. Here's where we try.

My theory is that we'll need to change both ingest and retrival methods.

## Ingest

Key findings so far:

* Normal resolution is set way too low -- it's _characters_. Up to 1k/10k
* Largest resolution isn't chunking right and is also set too low. Add `by_title`, chunk to 280k/300k
* Embedding service was getting overwhelmed. Added a rate limit and backoff to deal with it

In [ ]:
from functools import partial
from elasticsearch.helpers import scan, bulk

from redbox.loader.ingester import get_elasticsearch_store, get_elasticsearch_store_without_embeddings, UnstructuredLargeChunkLoader, UnstructuredTitleLoader
from redbox.loader.base import BaseRedboxFileLoader

from langchain_core.runnables import RunnableParallel, chain, RunnableLambda
from langchain.vectorstores import VectorStore

In [ ]:
from collections.abc import Iterator
from datetime import UTC, datetime

import requests
import tiktoken
from langchain_core.documents import Document

from redbox.models.file import ChunkResolution, ChunkMetadata
from redbox.loader.base import BaseRedboxFileLoader

encoding = tiktoken.get_encoding("cl100k_base")

class NewUnstructuredLargeChunkLoader(BaseRedboxFileLoader):
    """Load, partition and chunk a document using local unstructured library"""

    def lazy_load(self) -> Iterator[Document]:  # <-- Does not take any arguments
        """A lazy loader that reads a file line by line.

        When you're implementing lazy load methods, you should use a generator
        to yield documents one by one.
        """

        url = f"http://{self.env.unstructured_host}:8000/general/v0/general"
        files = {
            "files": (self.file_name, self.file_bytes),
        }
        response = requests.post(
            url,
            files=files,
            data={
                "chunking_strategy": "by_title",
                "strategy": "fast",
                "combine_under_n_chars": 280_000,
                "max_characters": 300_000,
                "overlap": self.env.worker_ingest_largest_chunk_overlap,
                "overlap_all": True,
            },
        )

        if response.status_code != 200:
            raise ValueError(response.text)

        elements = response.json()

        if not elements:
            raise ValueError("Unstructured failed to extract text for this file")

        for i, raw_chunk in enumerate(elements):
            yield Document(
                page_content=raw_chunk["text"],
                metadata=ChunkMetadata(
                    index=i,
                    file_name=raw_chunk["metadata"].get("filename"),
                    page_number=raw_chunk["metadata"].get("page_number"),
                    created_datetime=datetime.now(UTC),
                    token_count=len(encoding.encode(raw_chunk["text"])),
                    chunk_resolution=ChunkResolution.largest,
                ).model_dump(),
            )

In [ ]:
import backoff
import time
from requests.exceptions import HTTPError
from datetime import datetime, timedelta

class RateLimiter:
    def __init__(self, requests_per_minute):
        self.requests_per_minute = requests_per_minute
        self.reset_time = datetime.now()
        self.request_count = 0

    def check_rate_limit(self):
        now = datetime.now()
        if now >= self.reset_time + timedelta(minutes=1):
            # Reset the counter after 1 minute
            self.request_count = 0
            self.reset_time = now
        if self.request_count >= self.requests_per_minute:
            # Wait until 1 minute is over
            sleep_time = (self.reset_time + timedelta(minutes=1) - now).total_seconds()
            time.sleep(sleep_time)
            self.request_count = 0
            self.reset_time = datetime.now()

        self.request_count += 1


def build_local_document_loader(document_loader_type: type[BaseRedboxFileLoader], env: Settings):
    @chain
    def _loader(file_path: Path):
        return document_loader_type(
            file_name=file_path.name, 
            file_bytes=file_path.open("br").read(), 
            env=env
        ).lazy_load()
    return _loader


def build_add_documents_with_backoff(vectorstore: VectorStore):
    rate_limiter = RateLimiter(requests_per_minute=3_000)


    @chain
    @backoff.on_exception(
        backoff.expo, 
        HTTPError, 
        max_tries=5, 
        giveup=lambda e: e.response.status_code not in [429, 500, 503]
    )
    def _add_documents_with_backoff(documents: list[Document]) -> list[str]:
        ids: list[str] = []
        for doc in documents:
            # Check and enforce rate limit
            rate_limiter.check_rate_limit()
            
            try:
                id = vectorstore.add_documents([doc], create_index_if_not_exists=False)
                ids.append(id)
            except HTTPError as e:
                if e.response.status_code in [429, 500, 503]:
                    logging.warning(f"Retrying due to HTTPError {e.response.status_code}")
                    raise
                else:
                    raise
        return ids
    
    return _add_documents_with_backoff


def ingest_from_bytes(document_loader_type: type[BaseRedboxFileLoader], vectorstore: VectorStore, env: Settings):
    return (
        build_local_document_loader(
            document_loader_type=document_loader_type, 
            env=env
        )
        | RunnableLambda(list)
        | build_add_documents_with_backoff(vectorstore)
    )

In [ ]:
def ingest_file(file_path: Path) -> str | None:
    logging.info("Ingesting file: %s", file_path.name)

    es = ENV.elasticsearch_client()
    es_index_name = TEST_INDEX

    es.indices.create(index=es_index_name, ignore=[400])

    chunk_ingest_chain = ingest_from_bytes(
        document_loader_type=UnstructuredTitleLoader,
        vectorstore=get_elasticsearch_store(es, es_index_name),
        env=ENV,
    )

    large_chunk_ingest_chain = ingest_from_bytes(
        document_loader_type=NewUnstructuredLargeChunkLoader,
        vectorstore=get_elasticsearch_store_without_embeddings(es, es_index_name),
        env=ENV,
    )

    try:
        new_ids = RunnableParallel({
            "normal": chunk_ingest_chain, 
            "largest": large_chunk_ingest_chain
        }).invoke(file_path)

        logging.info("File: %s %s chunks ingested", file_path.name, {k: len(v) for k, v in new_ids.items()})

    except Exception as e:
        logging.exception("Error while processing file [%s]", file_path.name)
        
        return f"{type(e)}: {e.args[0]}"


In [ ]:
for f in (Path.cwd().parents[2] / "Downloads/DS").glob("*.pdf"):
    ingest_file(file_path=f)

In [ ]:
for f in (Path.cwd().parents[2] / "Downloads/Giant").glob("*.txt"):
    ingest_file(file_path=f)

In [ ]:
# Clear index function
def clear_index(index: str, env: Settings) -> None:
    es = env.elasticsearch_client()
    documents = scan(es, index=index, query={"query": {"match_all": {}}})
    bulk_data = [
        {"_op_type": "delete", "_index": doc['_index'], "_id": doc['_id']} for doc in documents
    ]
    bulk(es, bulk_data, request_timeout=300)

clear_index(TEST_INDEX, ENV)

## Retrieval

Key findings so far

* "What does the bible say about death" doesn't return chunks from the Bible

### Ideas

* Get the LLM to write some metadata filters
* Use an LLM to summarise what each document is to help the LLM write metadata filters for what it needs

In [ ]:
from uuid import UUID, uuid4

from langgraph.graph import START, END, StateGraph

from redbox.graph.nodes.processes import (
    PromptSet,
)
from redbox.models.chain import RedboxState, RedboxQuery, ChainChatMessage, AISettings
from redbox.chains.components import get_embeddings
from redbox.models.chat import ChatRoute
from redbox.graph.nodes.processes import (
    build_chat_pattern,
    build_set_route_pattern,
    build_retrieve_pattern,
    build_stuff_pattern,
)
from redbox.retriever import AllElasticsearchRetriever, ParameterisedElasticsearchRetriever

In [ ]:
retriever = ParameterisedElasticsearchRetriever(
    es_client=ENV.elasticsearch_client(),
    index_name=TEST_INDEX,
    embedding_model=get_embeddings(ENV),
    embedding_field_name=ENV.embedding_document_field_name,
)

In [ ]:
builder = StateGraph(RedboxState)

# Processes
builder.add_node("p_set_search_route", build_set_route_pattern(route=ChatRoute.search))
builder.add_node("p_condense_question", build_chat_pattern(prompt_set=PromptSet.CondenseQuestion))
builder.add_node("p_retrieve_docs", build_retrieve_pattern(retriever=retriever, final_source_chain=True))
builder.add_node("p_stuff_docs", build_stuff_pattern(prompt_set=PromptSet.Search, final_response_chain=True))

# Edges
builder.add_edge(START, "p_set_search_route")
builder.add_edge("p_set_search_route", "p_condense_question")
builder.add_edge("p_condense_question", "p_retrieve_docs")
builder.add_edge("p_retrieve_docs", "p_stuff_docs")
builder.add_edge("p_stuff_docs", END)

app = builder.compile(debug=False)

In [ ]:
state = RedboxState(
    request=RedboxQuery(
        question="What does the bible say about death?",
        s3_keys=[
            'warandpeace.txt', 
            'bible.txt', 
            'insearchoflosttime.txt',
            'KAN- Kolmogorov–Arnold Networks.pdf',
            'Mamba- Linear-Time Sequence Modeling with Selective State Spaces.pdf'
        ],
        user_uuid=uuid4(),
        chat_history=[],
        ai_settings=AISettings(),
    ),
)

app.invoke(state)